# 🏏 PSL ML Project
## Notebook 3: Preprocessing & Feature Engineering
Yahan hum raw ball-by-ball data ko ML-ready format mein convert karein ge.
**Target:** Predict karo ki **match mein winning team kaun hogi** (binary classification)

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('E:/PSL_Predictor/data/raw/psl_data.csv')
print('✅ Data loaded! Shape:', df.shape)

✅ Data loaded! Shape: (73784, 27)


### Step 1: Date Column Fix

In [5]:
# Date column ko proper datetime mein convert karo
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
print('Date range:', df['date'].min(), 'to', df['date'].max())
print('Sample dates:', df['date'].head().tolist())

Date range: 2016-04-02 00:00:00 to 2025-12-04 00:00:00
Sample dates: [Timestamp('2017-09-02 00:00:00'), Timestamp('2017-09-02 00:00:00'), Timestamp('2017-09-02 00:00:00'), Timestamp('2017-09-02 00:00:00'), Timestamp('2017-09-02 00:00:00')]


### Step 2: Match-Level Features Banana

In [6]:
# Har match ke liye aggregate features banana
# Hum 1st innings complete hone ke baad 2nd innings ki prediction karein ge

def get_match_features(group):
    """Ek match ke liye features calculate karo"""
    inn1 = group[group['inning'] == 1]
    inn2 = group[group['inning'] == 2]
    
    if inn1.empty or inn2.empty:
        return None
    
    batting_team_inn1 = inn1['batting_team'].iloc[0]
    bowling_team_inn1 = inn1['bowling_team'].iloc[0]
    winner = group['winner'].iloc[0]
    
    # Target: 1 agar 1st innings batting team jeete, 0 agar doosri team jeete
    target = 1 if winner == batting_team_inn1 else 0
    
    features = {
        'match_id': group['match_id'].iloc[0],
        'season': group['season'].iloc[0],
        'venue': group['venue'].iloc[0],
        'batting_team_inn1': batting_team_inn1,
        'bowling_team_inn1': bowling_team_inn1,
        
        # 1st innings stats
        'inn1_total_runs': inn1['total_runs'].sum(),
        'inn1_wickets': inn1['is_wicket'].sum(),
        'inn1_extras': inn1['extra_runs'].sum(),
        'inn1_boundaries': ((inn1['batsman_runs']==4) | (inn1['batsman_runs']==6)).sum(),
        'inn1_sixes': (inn1['batsman_runs']==6).sum(),
        'inn1_fours': (inn1['batsman_runs']==4).sum(),
        'inn1_dot_balls': (inn1['batsman_runs']==0).sum(),
        
        # 1st innings powerplay (overs 1-6)
        'inn1_pp_runs': inn1[inn1['over'] < 6]['total_runs'].sum(),
        'inn1_pp_wickets': inn1[inn1['over'] < 6]['is_wicket'].sum(),
        
        # 1st innings death overs (16-20)
        'inn1_death_runs': inn1[inn1['over'] >= 15]['total_runs'].sum(),
        'inn1_death_wickets': inn1[inn1['over'] >= 15]['is_wicket'].sum(),
        
        # 2nd innings (ye target ke liye)
        'inn2_total_runs': inn2['total_runs'].sum(),
        'inn2_wickets': inn2['is_wicket'].sum(),
        
        'winner': winner,
        'target': target
    }
    return features

# Har match pe apply karo
print('Match features compute ho rahi hain...')
match_features_list = []
for match_id, group in df.groupby('match_id'):
    feat = get_match_features(group)
    if feat:
        match_features_list.append(feat)

match_df = pd.DataFrame(match_features_list)
print(f'✅ Match-level dataset ready: {match_df.shape}')
match_df.head()

Match features compute ho rahi hain...
✅ Match-level dataset ready: (311, 20)


,match_id,season,venue,batting_team_inn1,bowling_team_inn1,inn1_total_runs,inn1_wickets,inn1_extras,inn1_boundaries,inn1_sixes,inn1_fours,inn1_dot_balls,inn1_pp_runs,inn1_pp_wickets,inn1_death_runs,inn1_death_wickets,inn2_total_runs,inn2_wickets,winner,target
0,959175,2016,Dubai International Cricket Stadium,Islamabad United,Quetta Gladiators,128,7,4,14,4,10,63,21,1,66,2,132,2,Quetta Gladiators,0
1,959177,2016,Dubai International Cricket Stadium,Lahore Qalandars,Karachi Kings,125,8,9,11,2,9,54,37,2,45,4,131,3,Karachi Kings,0
2,959179,2016,Dubai International Cricket Stadium,Peshawar Zalmi,Islamabad United,145,7,17,13,3,10,54,31,1,57,4,121,9,Peshawar Zalmi,1
3,959181,2016,Dubai International Cricket Stadium,Karachi Kings,Quetta Gladiators,147,7,11,12,3,9,46,29,1,63,3,153,2,Quetta Gladiators,0
4,959183,2016,Dubai International Cricket Stadium,Lahore Qalandars,Peshawar Zalmi,117,6,6,12,1,11,61,20,3,54,2,118,1,Peshawar Zalmi,0


### Step 3: Categorical Encoding

In [7]:
from sklearn.preprocessing import LabelEncoder

match_df_encoded = match_df.copy()

# Label encoding for categorical columns
le_team = LabelEncoder()
le_venue = LabelEncoder()

all_teams = pd.concat([match_df['batting_team_inn1'], match_df['bowling_team_inn1']]).unique()
le_team.fit(all_teams)

match_df_encoded['batting_team_enc'] = le_team.transform(match_df_encoded['batting_team_inn1'])
match_df_encoded['bowling_team_enc'] = le_team.transform(match_df_encoded['bowling_team_inn1'])
match_df_encoded['venue_enc'] = le_venue.fit_transform(match_df_encoded['venue'])

print('Teams encoded:', dict(zip(le_team.classes_, le_team.transform(le_team.classes_))))
print('\nVenues encoded:', dict(zip(le_venue.classes_, le_venue.transform(le_venue.classes_))))

Teams encoded: {'Islamabad United': np.int64(0), 'Karachi Kings': np.int64(1), 'Lahore Qalandars': np.int64(2), 'Multan Sultans': np.int64(3), 'Peshawar Zalmi': np.int64(4), 'Quetta Gladiators': np.int64(5)}

Venues encoded: {'Dubai International Cricket Stadium': np.int64(0), 'Gaddafi Stadium': np.int64(1), 'Gaddafi Stadium, Lahore': np.int64(2), 'Multan Cricket Stadium': np.int64(3), 'National Stadium': np.int64(4), 'National Stadium, Karachi': np.int64(5), 'Rawalpindi Cricket Stadium': np.int64(6), 'Sharjah Cricket Stadium': np.int64(7), 'Sheikh Zayed Stadium': np.int64(8), 'Sheikh Zayed Stadium, Abu Dhabi': np.int64(9)}


### Step 4: Derived Features

In [8]:
# Extra meaningful features banana
match_df_encoded['inn1_run_rate'] = (match_df_encoded['inn1_total_runs'] / 20).round(2)
match_df_encoded['inn1_boundary_pct'] = (
    (match_df_encoded['inn1_boundaries'] / 120 * 100)).round(2)  # 120 balls per innings approx
match_df_encoded['inn1_dot_ball_pct'] = (
    match_df_encoded['inn1_dot_balls'] / 120 * 100).round(2)
match_df_encoded['pp_run_rate'] = (match_df_encoded['inn1_pp_runs'] / 6).round(2)
match_df_encoded['death_run_rate'] = (match_df_encoded['inn1_death_runs'] / 5).round(2)

print('New features added!')
print(match_df_encoded.columns.tolist())

New features added!
['match_id', 'season', 'venue', 'batting_team_inn1', 'bowling_team_inn1', 'inn1_total_runs', 'inn1_wickets', 'inn1_extras', 'inn1_boundaries', 'inn1_sixes', 'inn1_fours', 'inn1_dot_balls', 'inn1_pp_runs', 'inn1_pp_wickets', 'inn1_death_runs', 'inn1_death_wickets', 'inn2_total_runs', 'inn2_wickets', 'winner', 'target', 'batting_team_enc', 'bowling_team_enc', 'venue_enc', 'inn1_run_rate', 'inn1_boundary_pct', 'inn1_dot_ball_pct', 'pp_run_rate', 'death_run_rate']


### Step 5: Save Processed Data

In [ ]:
# Save processed match-level dataset
match_df_encoded.to_csv('E:/PSL_Predictor/data/psl_match_features.csv', index=False)
print('✅ Processed data saved to ../data/psl_match_features.csv')
print(f'Total matches: {len(match_df_encoded)}')
print(f'Class balance (1st innings team wins):')
print(match_df_encoded['target'].value_counts(normalize=True).round(3))

OSError: Cannot save file into a non-existent directory: '..\data'

In [ ]:
# Correlation heatmap
import matplotlib.pyplot as plt
import seaborn as sns

num_features = ['inn1_total_runs', 'inn1_wickets', 'inn1_pp_runs', 'inn1_pp_wickets',
                'inn1_death_runs', 'inn1_death_wickets', 'inn1_sixes', 'inn1_fours',
                'inn1_dot_balls', 'inn1_run_rate', 'target']

corr = match_df_encoded[num_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plots/07_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved!')

## ✅ Feature Engineering Complete!
- Ball-by-ball data ko match-level features mein convert kiya
- Powerplay, death overs, boundaries, wickets sab features hain
- Categorical encoding ho gayi
- **Next:** Model Training notebook!